In [ ]:
# -------------------------------------------------
# Initialize
# -------------------------------------------------
import sys, os, re
import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal

In [ ]:
# -------------------------------------------------
# Define Data Directory
# -------------------------------------------------
AnalysisDir = 'Analysis'
DataDir = 'Data'

csvMIC = f'LMIC_Isolates_stMIC.csv'
csvAMR = f'LMIC_Isolates_stAMR.csv'
csvAST = f'LMIC_Isolates_stAST.csv'

# -------------------------------------------------
OutDir = os.path.join(AnalysisDir)
if not os.path.isdir(OutDir):
    os.mkdir(OutDir)

In [ ]:
#---------------------------------------------------
# Get BMD MIC/BP data
#---------------------------------------------------
dfMIC = pd.read_csv(os.path.join(DataDir,csvMIC))
dfMIC.rename(columns={"BMD_MIC": "MIC", "BMD_BP": "BP"}, inplace=True)
#dfMIC = allMIC[allMIC['Paper']=='Y']
dfMIC.columns = [c.strip() for c in dfMIC.columns]
dfMIC = dfMIC[dfMIC["DRUG_NAME"] != "-"].copy()
dfMIC["BP"] = dfMIC["BP"].fillna("").str.strip()
dfMIC.loc[dfMIC["BP"] == "-", "BP"] = ""

# Combine Ab's and Kp's
dfMIC.loc[dfMIC['ORGANISM_NAME'] == 'Acinetobacter baumannii complex', 'ORGANISM_NAME'] = 'Acinetobacter baumannii'
dfMIC.loc[dfMIC['ORGANISM_NAME'] == 'Klebsiella quasipneumoniae', 'ORGANISM_NAME'] = 'Klebsiella pneumoniae'

# collapse combo classes (e.g. "Beta-lactam|BLI" -> "Beta-lactam") for readability
dfMIC["CLASS_NORM"] = dfMIC["DRUG_CLASS"].str.split("|").str[0]
dfMIC["is_R"] = (dfMIC["BP"] == "R").astype(int)

dfBP = dfMIC[dfMIC["BP"] != ""].copy()
dfBP["is_R"] = (dfBP["BP"] == "R").astype(int)

# Calc log2(MIC)
def parse_mic_to_log2(raw):
    if pd.isna(raw):
        return np.nan
    first = raw.split("|")[0].strip()
    m = re.match(r"^(<=|>=|<|>)?(\d+\.?\d*)$", first)
    if not m:
        return np.nan
    op, num = m.groups()
    val = float(num)
    if op == ">":
        val *= 2
    return np.log2(val)
dfMIC["log2_MIC"] = dfMIC["MIC"].apply(parse_mic_to_log2)


#---------------------------------------------------
# Get Vitek MIC/BP data
#---------------------------------------------------
dfAST = pd.read_csv(os.path.join(DataDir,csvAST))
#dfMIC.rename(columns={"BMD_MIC": "MIC", "BMD_BP": "BP"}, inplace=True)
#dfAST = allAST[allAST['Paper']=='Y']
dfAST.columns = [c.strip() for c in dfAST.columns]
dfAST = dfAST[dfAST["DRUG_NAME"] != "-"].copy()
dfAST["VITEK_BP"] = dfAST["VITEK_BP"].fillna("").str.strip()
dfAST.loc[dfAST["VITEK_BP"] == "-", "VITEK_BP"] = ""

# Combine Ab's and Kp's
dfAST.loc[dfAST['ORGANISM_NAME'] == 'Acinetobacter baumannii complex', 'ORGANISM_NAME'] = 'Acinetobacter baumannii'
dfAST.loc[dfAST['ORGANISM_NAME'] == 'Klebsiella quasipneumoniae', 'ORGANISM_NAME'] = 'Klebsiella pneumoniae'

In [ ]:
# ---------------------------------------------------------------------------
# IS POLYMYXIN RESISTANCE LINKED TO MDR / XDR / PDR STATUS?
# ---------------------------------------------------------------------------
DRUG_CATEGORY = {
    "Amikacin": "Aminoglycosides", "Gentamicin": "Aminoglycosides",
    "Netilmicin": "Aminoglycosides", "Tobramycin": "Aminoglycosides",
    "Streptomycin": "Aminoglycosides", "Spectinomycin": "Aminoglycosides",
    "Doripenem": "Carbapenems", "Ertapenem": "Carbapenems",
    "Imipenem": "Carbapenems", "Meropenem": "Carbapenems",
    "Cefepime": "Cephalosporins", "Cefotaxime": "Cephalosporins",
    "Cefoxitin": "Cephalosporins", "Ceftaroline": "Cephalosporins",
    "Ceftriaxone": "Cephalosporins", "Cefuroxime": "Cephalosporins",
    "Cefazolin": "Cephalosporins", "Cefalotin": "Cephalosporins",
    "Ampicillin": "Penicillins", "Piperacillin": "Penicillins",
    "Ticarcillin": "Penicillins", "Carbenicillin": "Penicillins",
    "Penicillin G": "Penicillins",
    "Amoxicillin|Clavulanic acid": "Penicillin+BLI combinations",
    "Ampicillin|Sulbactam": "Penicillin+BLI combinations",
    "Piperacillin|Tazobactam": "Penicillin+BLI combinations",
    "Ticarcillin|Clavulanic acid": "Penicillin+BLI combinations",
    "Aztreonam": "Monobactams",
    "Ciprofloxacin": "Fluoroquinolones", "Levofloxacin": "Fluoroquinolones",
    "Moxifloxacin": "Fluoroquinolones", "Ofloxacin": "Fluoroquinolones",
    "Nalidixic acid": "Fluoroquinolones",
    "Trimethoprim|Sulfamethoxazole": "Folate pathway inhibitors",
    "Tetracycline": "Tetracyclines", "Doxycycline": "Tetracyclines",
    "Minocycline": "Tetracyclines", "Tigecycline": "Glycylcyclines",
    "Fosfomycin": "Phosphonic acids", "Colistin": "Polymyxins",
    "Polymyxin B": "Polymyxins", "Chloramphenicol": "Phenicols",
    "Azithromycin": "Macrolides", "Nitrofurantoin": "Nitrofurans",
}
dfMIC["DRUG_CATEGORY"] = dfMIC["DRUG_NAME"].map(DRUG_CATEGORY)

def classify_isolate(g):
    tested = g[g["DRUG_CATEGORY"].notna() & (g["BP"] != "")]
    n_cat_tested = tested["DRUG_CATEGORY"].nunique()
    cats_R = tested.loc[tested["BP"] == "R", "DRUG_CATEGORY"].unique()
    n_cat_R = len(cats_R)
    fully_R_cats = sum((sub["BP"] == "R").all() for _, sub in tested.groupby("DRUG_CATEGORY"))
    if n_cat_tested == 0:
        status = "Not enough data"
    elif n_cat_R >= 3 and (n_cat_tested - n_cat_R) <= 2:
        status = "XDR" if fully_R_cats < n_cat_tested else "PDR"
    elif n_cat_R >= 3:
        status = "MDR"
    else:
        status = "Non-MDR"
    return pd.Series({
        "ORGANISM_NAME": g["ORGANISM_NAME"].iloc[0],
        "n_categories_tested": n_cat_tested,
        "n_categories_resistant": n_cat_R,
        "MDR_STATUS": status,
    })


# Isolate Level
isolate_level = dfMIC.groupby("ORGANISM_ID").apply(classify_isolate).reset_index()

# polymyxin resistance flag per isolate (Colistin and/or Polymyxin B tested R)
poly = dfMIC[dfMIC["DRUG_NAME"].isin(["Colistin", "Polymyxin B"])].copy()
poly_status = (
    poly[poly["BP"] != ""]
    .groupby("ORGANISM_ID")["BP"]
    .apply(lambda s: "Resistant" if (s == "R").any() else "Susceptible")
)
isolate_level = isolate_level.merge(
    poly_status.rename("POLYMYXIN_STATUS"), on="ORGANISM_ID", how="left"
)

print(isolate_level["MDR_STATUS"].value_counts())
isolate_level.to_csv(os.path.join(OutDir,f'MDR_XDR_Isolates.csv'), index=False)

In [ ]:
# ---------------------------------------------------------------------------
# DISTRIBUTION OF MDR/XDR/PDR PER ORGANISM
# ---------------------------------------------------------------------------

img_file = os.path.join(OutDir,f'MDR_by_Organism.png')
print(f" Image File: {img_file}")

TOP_N_ORG = 12
#top_orgs = isolate_level["ORGANISM_NAME"].value_counts().head(TOP_N_ORG).index
top_orgs = isolate_level["ORGANISM_NAME"].value_counts()
plot_df = isolate_level.copy()

# plot_df.loc[plot_df['ORGANISM_NAME'] == 'Acinetobacter baumannii complex', 'ORGANISM_NAME'] = 'Acinetobacter baumannii'
# plot_df.loc[plot_df['ORGANISM_NAME'] == 'Klebsiella quasipneumoniae', 'ORGANISM_NAME'] = 'Klebsiella pneumoniae'

status_order = ["Non-MDR", "MDR", "XDR", "PDR"]
ct = (
    pd.crosstab(plot_df["ORGANISM_NAME"], plot_df["MDR_STATUS"], normalize="index") * 100
).reindex(columns=status_order, fill_value=0)

ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]  # keep top_orgs order by count already
ct = ct.sort_values(by='ORGANISM_NAME', ascending=False)

fig, ax = plt.subplots(figsize=(11, 7))
ct.plot(kind="barh", stacked=True, ax=ax,
        color=["#8fbf8f", "#f2c14e", "#e07a5f", "#6d0e0e"])
ax.set_xlabel(r"% of isolates", fontsize=16)
ax.set_ylabel("")
ax.set_title(f"MDR/XDR/PDR Distribution by Organism")
#plt.legend(fontsize='large', title_fontsize='16') 
ax.legend(title="Status", bbox_to_anchor=(1.02, 1), loc="upper left",fontsize=16)
for label in ax.get_yticklabels():
    label.set_style("italic")
ax.tick_params(axis='both', labelsize=14)
plt.tight_layout()
plt.savefig(img_file, dpi=300)
plt.show()
plt.close()

In [ ]:
#---------------------------------------------------
# MDR/XDR/PDR by #Isolates for each Organism
#---------------------------------------------------
ctN = (
    pd.crosstab(plot_df["ORGANISM_NAME"], plot_df["MDR_STATUS"]) 
).reindex(columns=status_order, fill_value=0)

ctN = ctN.loc[ctN.sum(axis=1).sort_values(ascending=False).index]  # keep top_orgs order by count already
ctN = ctN.sort_values(by='ORGANISM_NAME', ascending=True)

ctN

In [ ]:
#---------------------------------------------------
# MDR/XDR/PDR by Colistin/Polymyxin B Resistance Status
#---------------------------------------------------

tested_poly = isolate_level.dropna(subset=["POLYMYXIN_STATUS"]).copy()
tested_poly = tested_poly[tested_poly["MDR_STATUS"] != "Not enough data"]

print(f"\nIsolates with an interpreted Colistin/Polymyxin B result: {len(tested_poly)} / {len(isolate_level)}")

# --- 1. % polymyxin-resistant within each MDR status group ---
ct_counts = pd.crosstab(tested_poly["MDR_STATUS"], tested_poly["POLYMYXIN_STATUS"])
ct_pct = pd.crosstab(tested_poly["MDR_STATUS"], tested_poly["POLYMYXIN_STATUS"], normalize="index") * 100
status_order = [s for s in ["Non-MDR", "MDR", "XDR", "PDR"] if s in ct_pct.index]
ct_counts = ct_counts.reindex(status_order)
ct_pct = ct_pct.reindex(status_order)

# --- 2. overall resistance burden (n_categories_resistant) by polymyxin status ---
res_poly = tested_poly.loc[tested_poly["POLYMYXIN_STATUS"] == "Resistant", "n_categories_resistant"]
sus_poly = tested_poly.loc[tested_poly["POLYMYXIN_STATUS"] == "Susceptible", "n_categories_resistant"]
u_stat, p_mwu = mannwhitneyu(res_poly, sus_poly, alternative="greater")
print(f"\nMean categories resistant | Polymyxin-R: {res_poly.mean():.2f} (n={len(res_poly)}) "
      f"vs Polymyxin-S: {sus_poly.mean():.2f} (n={len(sus_poly)})")
print(f"Mann-Whitney U (one-sided, polymyxin-R > polymyxin-S): U={u_stat:.0f}, p={p_mwu:.2e}")

ct_pct

In [ ]:
#---------------------------------------------------
# PmxB vs COL
#---------------------------------------------------

img_file = os.path.join(OutDir,f'PmxB_vs_COL_by_Organism.png')
print(f" Image File: {img_file}")

poly = dfMIC[dfMIC["DRUG_NAME"].isin(["Colistin", "Polymyxin B"])].copy()
wide_mic = poly.pivot_table(index="ORGANISM_ID", columns="DRUG_NAME", values="log2_MIC", aggfunc="first")
meta = dfMIC.drop_duplicates("ORGANISM_ID").set_index("ORGANISM_ID")["ORGANISM_NAME"]
wide_mic = wide_mic.join(meta)
paired = wide_mic.dropna(subset=["Colistin", "Polymyxin B"]).copy()
paired["log2_fold_diff"] = paired["Colistin"] - paired["Polymyxin B"]
paired.sort_values(by='ORGANISM_NAME', ascending=False)

fig, ax = plt.subplots(figsize=(11, 7))

sns.boxplot(data=paired, y="ORGANISM_NAME", x="log2_fold_diff",
            hue="ORGANISM_NAME", palette="coolwarm", legend=False, ax=ax)
sns.stripplot(data=paired, y="ORGANISM_NAME", x="log2_fold_diff", 
              color="black", alpha=0.4, size=4, ax=ax)


# sns.boxplot(data=paired, x="ORGANISM_NAME", y="log2_fold_diff", order=order,
#             hue="ORGANISM_NAME", palette="coolwarm", legend=False, ax=ax)
# sns.stripplot(data=paired, x="ORGANISM_NAME", y="log2_fold_diff", order=order.sort(),
#               color="black", alpha=0.4, size=4, ax=ax)

#ax.ayhline(0, color="black", linewidth=1)
ax.set_xlabel("Log2(MIC[COL]) - Log2(MIC[PmxB])", fontsize=16)
ax.set_ylabel("")
for label in ax.get_yticklabels():
    label.set_style("italic")
ax.tick_params(axis='both', labelsize=14)
#ax.set_xlabel('X Label', )
#ax.set_title("Colistin vs Polymyxin B: paired MIC difference, by organism")
#plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(img_file, dpi=300)
plt.show()
plt.close()

In [ ]:
#---------------------------------------------------
# Get AMR data
#---------------------------------------------------

dfAMR = pd.read_csv(os.path.join(DataDir,csvAMR))

dfAMR.columns = [c.strip() for c in dfAMR.columns]
dfAMR = dfAMR.drop(columns=[c for c in dfAMR.columns if c.startswith("Unnamed")])
dfAMR = dfAMR[dfAMR["GENE_CODE"] != "-"].copy()

# Combine Ab's and Kp's
dfAMR.loc[dfAMR['ORGANISM_NAME'] == 'Acinetobacter baumannii complex', 'ORGANISM_NAME'] = 'Acinetobacter baumannii'
dfAMR.loc[dfAMR['ORGANISM_NAME'] == 'Klebsiella quasipneumoniae', 'ORGANISM_NAME'] = 'Klebsiella pneumoniae'

# AMR Genes+Mutations
res_genmut = dfAMR[((dfAMR["GENE_SUBTYPE"] == "AMR-Gene") | (dfAMR["GENE_SUBTYPE"] == "AMR-Mutation")) & dfAMR["AMR_CLASS"].notna()].copy()
# split compound classes like "Phenicol; Quinolone" into separate class memberships
res_genmut["AMR_CLASS_LIST"] = res_genmut["AMR_CLASS"].str.split(";").apply(lambda lst: [x.strip() for x in lst])
res_genmut_exploded = res_genmut.explode("AMR_CLASS_LIST")

genmut_matrix = (
    res_genmut_exploded.assign(present=1)
    .pivot_table(index=["ORGANISM_ID","ORGANISM_NAME"], columns="AMR_CLASS_LIST", values="present",
                 aggfunc="max", fill_value=0)
)


# AMR Mutations
res_mut = dfAMR[(dfAMR["GENE_SUBTYPE"] == "AMR-Mutation") & dfAMR["AMR_CLASS"].notna()].copy()
# split compound classes like "Phenicol; Quinolone" into separate class memberships
res_mut["AMR_CLASS_LIST"] = res_mut["AMR_CLASS"].str.split(";").apply(lambda lst: [x.strip() for x in lst])
res_mut_exploded = res_mut.explode("AMR_CLASS_LIST")

mut_matrix = (
    res_mut_exploded.assign(present=1)
    .pivot_table(index=["ORGANISM_ID","ORGANISM_NAME"], columns="AMR_CLASS_LIST", values="present",
                 aggfunc="max", fill_value=0)
)


# AMR Genes
res_gen = dfAMR[(dfAMR["GENE_SUBTYPE"] == "AMR-Gene") & dfAMR["AMR_CLASS"].notna()].copy()
# split compound classes like "Phenicol; Quinolone" into separate class memberships
res_gen["AMR_CLASS_LIST"] = res_gen["AMR_CLASS"].str.split(";").apply(lambda lst: [x.strip() for x in lst])
res_gen_exploded = res_gen.explode("AMR_CLASS_LIST")

gen_matrix = (
    res_gen_exploded.assign(present=1)
    .pivot_table(index=["ORGANISM_ID","ORGANISM_NAME"], columns="AMR_CLASS_LIST", values="present",
                 aggfunc="max", fill_value=0)
)


In [ ]:
#---------------------------------------------------
# phenotype: isolate x drug class, binary R (from BP)
#---------------------------------------------------

pheno_matrix = dfMIC.groupby(["ORGANISM_ID", "ORGANISM_NAME","DRUG_CATEGORY"])["is_R"].max().unstack("DRUG_CATEGORY")

# align on shared isolates
genmut_common_ids = genmut_matrix.index.intersection(pheno_matrix.index)
genmut_aligned = genmut_matrix.loc[genmut_common_ids]

mut_common_ids = mut_matrix.index.intersection(pheno_matrix.index)
mut_aligned = mut_matrix.loc[mut_common_ids]

gen_common_ids = gen_matrix.index.intersection(pheno_matrix.index)
gen_aligned = gen_matrix.loc[gen_common_ids]


pheno_aligned = pheno_matrix.loc[genmut_common_ids]

gen_aligned.to_csv(os.path.join(OutDir,f'AMRGene_Isolates.csv'), index=True)
genmut_aligned.to_csv(os.path.join(OutDir,f'AMRGeneMutation_Isolates.csv'), index=True)
mut_aligned.to_csv(os.path.join(OutDir,f'AMRMutation_Isolates.csv'), index=True)
pheno_aligned.to_csv(os.path.join(OutDir,f'Phenotype_Isolates.csv'), index=True)

In [ ]:
#---------------------------------------------------
# AMR Gene/Mutation by Organism
#---------------------------------------------------

img_file = os.path.join(OutDir,f'AMRGeneMutation_by_Organism.png')
print(f" Image File: {img_file}")

class_cols = [c for c in genmut_aligned.columns if c not in ("ORGANISM_ID", "ORGANISM_NAME")]

org_n = genmut_aligned.groupby("ORGANISM_NAME").size().rename("n_isolates")
counts_n = genmut_aligned.groupby("ORGANISM_NAME")[class_cols].sum()
data_pct = counts_n.div(org_n, axis=0) * 100


fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(data_pct, cmap="rocket_r", annot=True, fmt=".0f", cbar_kws={"label": r"% isolates"}, ax=ax)

ax.set_title("AMR Gene/Mutation Classes")
ax.set_xlabel("")
ax.set_ylabel("")
for label in ax.get_yticklabels():
    label.set_style("italic")

ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_xticklabels(
    ax.get_xticklabels(), 
    rotation=45, 
    horizontalalignment='left',
    rotation_mode='anchor'
)

plt.tight_layout()
plt.savefig(img_file, dpi=300)
plt.show()
plt.close()

In [ ]:
#---------------------------------------------------
# AMR Mutation by Organism
#---------------------------------------------------

img_file = os.path.join(OutDir,f'AMR_Mutation_by_Organism.png')
print(f" Image File: {img_file}")

class_cols = [c for c in mut_aligned.columns if c not in ("ORGANISM_ID", "ORGANISM_NAME")]

org_n = mut_aligned.groupby("ORGANISM_NAME").size().rename("n_isolates")
counts_n = mut_aligned.groupby("ORGANISM_NAME")[class_cols].sum()
data_pct = counts_n.div(org_n, axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(data_pct, cmap="rocket_r", annot=True, fmt=".0f", cbar_kws={"label": r"% isolates"}, ax=ax)

ax.set_title("AMR Mutation Classes")
ax.set_xlabel("")
ax.set_ylabel("")
for label in ax.get_yticklabels():
    label.set_style("italic")

ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_xticklabels(
    ax.get_xticklabels(), 
    rotation=45, 
    horizontalalignment='left',
    rotation_mode='anchor'
)

plt.tight_layout()
plt.savefig(img_file, dpi=300)
plt.show()
plt.close()

In [ ]:
#---------------------------------------------------
# AMR Gene by Organism
#---------------------------------------------------

img_file = os.path.join(OutDir,f'AMR_Gene_by_Organism.png')
print(f" Image File: {img_file}")

class_cols = [c for c in gen_aligned.columns if c not in ("ORGANISM_ID", "ORGANISM_NAME")]

org_n = gen_aligned.groupby("ORGANISM_NAME").size().rename("n_isolates")
counts_n = gen_aligned.groupby("ORGANISM_NAME")[class_cols].sum()
data_pct = counts_n.div(org_n, axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(data_pct, cmap="rocket_r", annot=True, fmt=".0f", cbar_kws={"label": r"% isolates"}, ax=ax)

ax.set_title("AMR Gene Classes")
ax.set_xlabel("")
ax.set_ylabel("")
for label in ax.get_yticklabels():
    label.set_style("italic")

ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_xticklabels(
    ax.get_xticklabels(), 
    rotation=45, 
    horizontalalignment='left',
    rotation_mode='anchor'
)

plt.tight_layout()
plt.savefig(img_file, dpi=300)
plt.show()
plt.close()

In [ ]:
#---------------------------------------------------
# Phenotype by Organism
#---------------------------------------------------

img_file = os.path.join(OutDir,f'Phenotype_by_Organism.png')
print(f" Image File: {img_file}")

class_cols = [c for c in pheno_aligned.columns if c not in ("ORGANISM_ID", "ORGANISM_NAME")]

org_n = pheno_aligned.groupby("ORGANISM_NAME").size().rename("n_isolates")
pheno_counts = pheno_aligned.groupby("ORGANISM_NAME")[class_cols].sum()
pheno_pct = pheno_counts.div(org_n, axis=0) * 100


fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(pheno_pct, cmap="rocket_r", annot=True, fmt=".0f", cbar_kws={"label": r"% isolates"}, ax=ax)

ax.set_title("Antibiotic Categories")
ax.set_xlabel("")
ax.set_ylabel("")
for label in ax.get_yticklabels():
    label.set_style("italic")
    
ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_xticklabels(
    ax.get_xticklabels(), 
    rotation=45, 
    horizontalalignment='left',
    rotation_mode='anchor'
)

plt.tight_layout()
plt.savefig(img_file, dpi=300)
plt.show()
plt.close()

In [ ]:
#---------------------------------------------------
# Phenotype/Genotype by Organism
#---------------------------------------------------

img_file = os.path.join(OutDir,f'Geno_Phenotype_Pooled.png')
print(f" Image File: {img_file}" )

geno_cols = [c for c in genmut_aligned.columns if c not in ("ORGANISM_ID", "ORGANISM_NAME")]
pheno_cols = [c for c in pheno_aligned.columns if c not in ("ORGANISM_ID", "ORGANISM_NAME")]

merged_aligned = genmut_aligned.merge(pheno_aligned, on=["ORGANISM_ID", "ORGANISM_NAME"], suffixes=("_geno", "_pheno"))

def corr_grid(df):
    grid = pd.DataFrame(index=geno_cols, columns=pheno_cols, dtype=float)
    n_grid = pd.DataFrame(index=geno_cols, columns=pheno_cols, dtype=float)
    for g in geno_cols:
        gcol = g if g not in pheno_cols else f"{g}_geno"
        gcol = g  # genotype columns are unambiguous vs pheno_cols namespace already
        for d in pheno_cols:
            dcol = d
            pair = df[[g, d]].dropna()
            if len(pair) >= 15 and pair[g].nunique() > 1 and pair[d].nunique() > 1:
                grid.loc[g, d] = pair.corr().iloc[0, 1]
                n_grid.loc[g, d] = len(pair)
    return grid, n_grid


# --- Pooled ---
pooled_grid, pooled_n = corr_grid(merged_aligned)
# print("\nPOOLED genotype x phenotype phi-coefficient grid:")
# print(pooled_grid.round(2).to_string())

fig, ax = plt.subplots(figsize=(12, 11))
sns.heatmap(pooled_grid.astype(float), cmap="coolwarm", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7.5}, ax=ax,
            cbar_kws={"label": "phi coefficient"})
ax.set_title("Phenotype class - Resistant")
#ax.set_xlabel("Phenotype class - Resistant")
ax.set_ylabel("Genotype class - Gene Present")

ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_xticklabels(
    ax.get_xticklabels(), 
    rotation=45, 
    horizontalalignment='left',
    rotation_mode='anchor',
)
ax.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.savefig(img_file, dpi=300)